In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tkinter as tk
from tkinter import filedialog
import mne
import pickle
import yaml

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from mne.decoding import CSP


In [2]:
def print_cm(cm, title, tick_labels):
    plt.figure(figsize=(4, 3))
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    ax.set_xticklabels(tick_labels)
    ax.set_yticklabels(tick_labels)
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title(title)
    plt.show()

def get_fisher_score(X, y, classes):
    mu1 = np.mean(X[y == classes[0]], axis=0)
    sigma1 = np.std(X[y == classes[0]], axis=0)
    mu2 = np.mean(X[y == classes[1]], axis=0)
    sigma2 = np.std(X[y == classes[1]], axis=0)
    fisher = np.abs(mu1 - mu2)**2 / (sigma1**2 + sigma2**2 + 1e-10)
    return fisher


In [7]:
# Interactive File Selection
root = tk.Tk()
root.withdraw() # Hide the main window
filenames = filedialog.askopenfilenames(title="Select GDF Files", filetypes=[("GDF Files", "*.gdf")])

if not filenames:
    print("No files selected!")
else:
    print(f"Selected {len(filenames)} files:")
    for f in filenames:
        print(f" - {f}")

    paradigm = os.path.splitext(os.path.basename(filenames[0]))[0].split('.')[-1]
    subject = os.path.splitext(os.path.basename(filenames[0]))[0].split('.')[0]
    
    # Determine classes based on filename
    if 'cvsa' in paradigm.lower():
        classes = ['730', '731']
    elif 'mi' in paradigm.lower():
        classes = ['769', '770']
    else:
        classes = ['771', '773']
        
    print(f"Paradigm: {paradigm}")
    print(f"Subject: {subject}")
    print(f"Classes: {classes}")


Selected 5 files:
 - /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.115210.calibration.mi_lhrh.gdf
 - /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.115631.calibration.mi_lhrh.gdf
 - /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.120226.calibration.mi_lhrh.gdf
 - /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.123549.calibration.mi_lhrh.gdf
 - /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.123954.calibration.mi_lhrh.gdf
Paradigm: mi_lhrh
Subject: c7
Classes: ['769', '770']


In [8]:
# Preprocessing and Epoching Parameters
TARGET_SFREQ = 128
CHUNK_SIZE = 8 # ~62.5 ms at 128Hz
WINDOW_SIZE = TARGET_SFREQ * 1 # 1 second window

BANDS = [[8, 10], [10, 12], [12, 14], [8, 14], [14, 20]]
BANDS_STR = [f"{b[0]}-{b[1]}Hz" for b in BANDS]

N_CHANNELS = 32
EXCLUDE_CHANNELS = ['Fp1', 'Fp2'] # Optionally exclude channels


In [9]:
raw_list = []
for f in filenames:
    raw = mne.io.read_raw_gdf(f, preload=True)
    raw = raw.pick(list(range(N_CHANNELS)))
    
    # Downsample
    raw.resample(TARGET_SFREQ)
    
    # Common Average Reference (CAR)
    raw.set_eeg_reference('average', projection=False)
    
    raw_list.append(raw)


Extracting EDF parameters from /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.115210.calibration.mi_lhrh.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
Fp1, Fz, F3, F7, F9, FC5, FC1, C3, T7, T9, CP5, CP1, Pz, P3, P7, O1, Oz, O2, P4, P8, T10, CP6, CP2, Cz, C4, T8, F10, FC6, FC2, F8, F4, Fp2, ACC_X, ACC_Y, ACC_Z
Creating raw.info structure...


/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Physical range is not defined in following channels:
ACC_X, ACC_Y, ACC_Z
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)


Reading 0 ... 114499  =      0.000 ...   228.998 secs...
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Extracting EDF parameters from /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.115631.calibration.mi_lhrh.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
Fp1, Fz, F3, F7, F9, FC5, FC1, C3, T7, T9, CP5, CP1, Pz, P3, P7, O1, Oz, O2, P4, P8, T10, CP6, CP2, Cz, C4, T8, F10, FC6, FC2, F8, F4, Fp2, ACC_X, ACC_Y, ACC_Z
Creating raw.info structure...
Reading 0 ... 115499  =      0.000 ...   230.998 secs...


/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Physical range is not defined in following channels:
ACC_X, ACC_Y, ACC_Z
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Extracting EDF parameters from /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.120226.calibration.mi_lhrh.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
Fp1, Fz, F3, F7, F9, FC5, FC1, C3, T7, T9, CP5, CP1, Pz, P3, P7, O1, Oz, O2, P4, P8, T10, CP6, CP2, Cz, C4, T8, F10, FC6, FC2, F8, F4, Fp2, ACC_X, ACC_Y, ACC_Z
Creating raw.info structure...
Reading 0 ... 113499  =      0.000 ...   226.998 secs...


...

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Extracting EDF parameters from /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.123549.calibration.mi_lhrh.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
Fp1, Fz, F3, F7, F9, FC5, FC1, C3, T7, T9, CP5, CP1, Pz, P3, P7, O1, Oz, O2, P4, P8, T10, CP6, CP2, Cz, C4, T8, F10, FC6, FC2, F8, F4, Fp2, ACC_X, ACC_Y, ACC_Z
Creating raw.info structure...
Reading 0 ... 111999  =      0.000 ...   223.998 secs...


/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Physical range is not defined in following channels:
ACC_X, ACC_Y, ACC_Z
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Extracting EDF parameters from /home/paolo/bci_vr_ws/recordings/c7/20260428/calibration/gdf/c7.20260428.123954.calibration.mi_lhrh.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
Fp1, Fz, F3, F7, F9, FC5, FC1, C3, T7, T9, CP5, CP1, Pz, P3, P7, O1, Oz, O2, P4, P8, T10, CP6, CP2, Cz, C4, T8, F10, FC6, FC2, F8, F4, Fp2, ACC_X, ACC_Y, ACC_Z
Creating raw.info structure...
Reading 0 ... 112499  =      0.000 ...   224.998 secs...


/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Physical range is not defined in following channels:
ACC_X, ACC_Y, ACC_Z
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)
/tmp/ipykernel_182531/1022400080.py:3: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_gdf(f, preload=True)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


In [30]:
# Extract 1-second sliding windows during continuous feedback (event 781)
X_epochs = []
y_epochs = []
trials = []
trial_idx = 0

all_X_band = [] # Will store band-filtered windows

for raw in raw_list:
    # Get annotations
    cf_segments = []
    current_class = None
    onset_sample = None # Variabile per salvare l'inizio del feedback
    
    for annot in raw.annotations:
        desc = annot['description']
        
        if desc in classes:
            current_class = desc
            
        elif desc == '781':
            onset_sample = int(annot['onset'] * TARGET_SFREQ)
            
        elif desc == '33549':
            if current_class is not None and onset_sample is not None:
                end_sample = int(annot['onset'] * TARGET_SFREQ)
                duration_samples = end_sample - onset_sample
                
                # Salviamo onset, duration calcolata e classe
                cf_segments.append((onset_sample, duration_samples, current_class))
                
                # Reset delle variabili per il trial successivo
                current_class = None
                onset_sample = None
                
    # Bandpass filter the raw data for each band
    raw_bands = []
    for l_freq, h_freq in BANDS:
        raw_b = raw.copy().filter(l_freq, h_freq, method='iir', iir_params={'order': 4, 'ftype': 'butter'})
        raw_bands.append(raw_b.get_data())
        
    for onset, duration, cls in cf_segments:
        end = onset + duration
        
        # Sliding window from onset to end - window_size, step by chunk_size
        for s in range(onset, end - WINDOW_SIZE + 1, CHUNK_SIZE):
            window_bands = []
            for b in range(len(BANDS)):
                window_bands.append(raw_bands[b][:, s : s + WINDOW_SIZE])
            X_epochs.append(window_bands)
            y_epochs.append(cls)
            trials.append(trial_idx)
        trial_idx += 1

X_epochs = np.array(X_epochs) # Shape: (n_epochs, n_bands, n_channels, n_samples)
y_epochs = np.array(y_epochs)
trials = np.array(trials)

print(f"Extracted {len(X_epochs)} windows across {len(np.unique(trials))} trials.")

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 10 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 8.00, 10.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 12 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 10.00, 12.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 14 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 12.00, 14.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up ban

In [11]:
# Fit CSP per band to compute Fisher Scores
csp_list = [CSP(n_components=N_CHANNELS, reg=None, log=True, norm_trace=False) for _ in range(len(BANDS))]

fisher_scores = np.zeros((len(BANDS), N_CHANNELS))
X_csp_all = np.zeros((len(X_epochs), len(BANDS), N_CHANNELS))

for b in range(len(BANDS)):
    X_b = X_epochs[:, b, :, :]
    csp_list[b].fit(X_b, y_epochs)
    X_csp_all[:, b, :] = csp_list[b].transform(X_b)
    
    # Compute Fisher score
    fisher_scores[b, :] = get_fisher_score(X_csp_all[:, b, :], y_epochs, classes)

# Plot Fisher Scores
plt.figure(figsize=(12, 4))
sns.heatmap(fisher_scores, annot=False, cmap='viridis', xticklabels=np.arange(1, N_CHANNELS+1), yticklabels=BANDS_STR)
plt.title('Fisher Score per Component and Band')
plt.xlabel('CSP Component (Sorted by Variance Ratio)')
plt.ylabel('Frequency Band')
plt.show()


IndexError: too many indices for array: array is 1-dimensional, but 4 were indexed

In [ ]:
# Select Top N and Bottom N components per band
N_COMPONENTS = 3 # Take top 3 and bottom 3
selected_indices = list(range(N_COMPONENTS)) + list(range(N_CHANNELS - N_COMPONENTS, N_CHANNELS))

X_features = np.zeros((len(X_epochs), len(BANDS) * len(selected_indices)))

for b in range(len(BANDS)):
    X_features[:, b * len(selected_indices) : (b+1) * len(selected_indices)] = X_csp_all[:, b, selected_indices]
    
print(f"Feature matrix shape: {X_features.shape}")


In [ ]:
# Train and Evaluate sLDA
# GroupShuffleSplit to avoid mixing chunks from the same trial into train/test
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss_val.split(X_features, y_epochs, groups=trials))

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss_test.split(X_features[temp_idx], y_epochs[temp_idx], groups=trials[temp_idx]))

# Map back to original indices
val_idx = temp_idx[val_idx]
test_idx = temp_idx[test_idx]

X_train, y_train = X_features[train_idx], y_epochs[train_idx]
X_val, y_val = X_features[val_idx], y_epochs[val_idx]
X_test, y_test = X_features[test_idx], y_epochs[test_idx]

# Train sLDA
clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
clf.fit(X_train, y_train)

# Evaluate
y_pred_train = clf.predict(X_train)
y_pred_val = clf.predict(X_val)
y_pred_test = clf.predict(X_test)

print(f"Train Balanced Accuracy: {balanced_accuracy_score(y_train, y_pred_train):.4f}")
print(f"Validation Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_val):.4f}")
print(f"Test Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_test):.4f}")

cm_test = confusion_matrix(y_test, y_pred_test, labels=classes)
print_cm(cm_test, "Test Set Confusion Matrix", classes)


In [ ]:
# Extract CSP matrices (Filters and Patterns)
csp_matrices = []
for b in range(len(BANDS)):
    csp = csp_list[b]
    filters = csp.filters_ # Spatial filters
    patterns = csp.patterns_ # Spatial patterns
    
    # Store only the selected components
    sel_filters = filters[selected_indices, :]
    csp_matrices.append(sel_filters.tolist())

# Save Python Pickle with all metadata
save_dict = {
    'model': clf,
    'classes': classes,
    'bands': BANDS,
    'selected_components_indices': selected_indices,
    'filenames': filenames,
    'train_acc': balanced_accuracy_score(y_train, y_pred_train),
    'val_acc': balanced_accuracy_score(y_val, y_pred_val),
    'test_acc': balanced_accuracy_score(y_test, y_pred_test),
    'csp_filters': csp_matrices
}

import time
time_str = time.strftime('%d%m%Y_%H%M%S')
pkl_filename = f'slda_model_{subject}_{time_str}.pkl'
with open(pkl_filename, 'wb') as f:
    pickle.dump(save_dict, f)
print(f"Saved model and metadata to {pkl_filename}")

# Save YAML file for ROS C++ Node with the CSP matrices
yaml_filename = f'csp_{subject}_{time_str}.yaml'
with open(yaml_filename, 'w') as f:
    yaml.dump({'csp_matrices': csp_matrices}, f)
print(f"Saved CSP matrices to {yaml_filename} for ROS node")
